# 03 — Gold: Análises de Negócio (SQL)

**Responsabilidade:** Criar views SQL no schema `gold` respondendo as perguntas do case.
Disponíveis no Databricks SQL Editor para o time de negócio.

> Execute `00_config` antes deste notebook.

## Célula 1 — Carregar configurações

In [ ]:
%run "./00_config"

## Célula 2 — Criar schema Gold

In [ ]:
spark.sql(f'CREATE SCHEMA IF NOT EXISTS {CATALOG}.gold')
print(f"Schema: {CATALOG}.gold")
spark.sql(f"SHOW SCHEMAS IN {CATALOG}").show()

## Célula 3 — ANÁLISE 1: Média de total_amount por mês

**Pergunta:** Qual a média de `total_amount` recebido em um mês considerando todos os yellow táxis da frota?

**Resultado esperado:**
| Mês | Média | Mediana |
|---|---|---|
| Janeiro | $27,46 | $20,16 |
| Fevereiro | $27,37 | $20,20 |
| Março | $28,29 | $20,62 |
| Abril | $28,78 | $20,93 |
| Maio | $29,45 | $21,36 |

In [ ]:
spark.sql(f'''
    CREATE OR REPLACE VIEW {CATALOG}.gold.vw_avg_total_amount_per_month AS
    SELECT
        year                                        AS ano,
        CAST(month AS INT)                          AS mes_numero,
        CASE CAST(month AS INT)
            WHEN 1 THEN 'Janeiro'   WHEN 2 THEN 'Fevereiro'
            WHEN 3 THEN 'Marco'     WHEN 4 THEN 'Abril'
            WHEN 5 THEN 'Maio'
        END                                         AS mes_nome,
        COUNT(*)                                    AS total_corridas,
        ROUND(AVG(total_amount), 2)                 AS media_total_amount,
        ROUND(MIN(total_amount), 2)                 AS min_total_amount,
        ROUND(MAX(total_amount), 2)                 AS max_total_amount,
        ROUND(PERCENTILE(total_amount, 0.5), 2)     AS mediana_total_amount,
        ROUND(STDDEV(total_amount), 2)              AS desvio_padrao
    FROM {SILVER_TABLE}
    WHERE year = '2023'
    GROUP BY year, month
    ORDER BY mes_numero
''')
print(f"View: {CATALOG}.gold.vw_avg_total_amount_per_month")
spark.sql(f"SELECT * FROM {CATALOG}.gold.vw_avg_total_amount_per_month").show(truncate=False)

## Célula 4 — ANÁLISE 2: Média de passageiros por hora em Maio

**Pergunta:** Qual a média de `passenger_count` por hora do dia em maio considerando todos os táxis da frota?

**Insight principal:** Pico de demanda entre 17h-19h com maior volume de corridas. Madrugada tem média levemente maior de passageiros por corrida (1,43-1,46).

In [ ]:
spark.sql(f'''
    CREATE OR REPLACE VIEW {CATALOG}.gold.vw_avg_passengers_per_hour_may AS
    SELECT
        HOUR(pickup_datetime)                       AS hora_do_dia,
        CASE
            WHEN HOUR(pickup_datetime) BETWEEN 0  AND 5  THEN '00-05 Madrugada'
            WHEN HOUR(pickup_datetime) BETWEEN 6  AND 11 THEN '06-11 Manha'
            WHEN HOUR(pickup_datetime) BETWEEN 12 AND 17 THEN '12-17 Tarde'
            WHEN HOUR(pickup_datetime) BETWEEN 18 AND 23 THEN '18-23 Noite'
        END                                         AS periodo_do_dia,
        COUNT(*)                                    AS total_corridas,
        ROUND(AVG(passenger_count), 2)              AS media_passageiros,
        SUM(passenger_count)                        AS total_passageiros
    FROM {SILVER_TABLE}
    WHERE year = '2023' AND month = '5'
    GROUP BY hora_do_dia
    ORDER BY hora_do_dia
''')
print(f"View: {CATALOG}.gold.vw_avg_passengers_per_hour_may")
spark.sql(f"SELECT * FROM {CATALOG}.gold.vw_avg_passengers_per_hour_may").show(24, truncate=False)

## Célula 5 — Verificar catálogo completo

In [ ]:
print('=' * 55)
print('CATALOGO: ifood_catalog (Unity Catalog)')
print('=' * 55)
for schema in ['bronze','silver','gold']:
    print(f"\n {CATALOG}.{schema}")
    spark.sql(f"SHOW TABLES IN {CATALOG}.{schema}").show(truncate=False)
print()
print('Pipeline completo!')
print(f"  {CATALOG}.bronze.yellow_taxi")
print(f"  {CATALOG}.silver.yellow_taxi")
print(f"  {CATALOG}.gold.vw_avg_total_amount_per_month")
print(f"  {CATALOG}.gold.vw_avg_passengers_per_hour_may")
print()
print('Acesse no SQL Editor:')
print(f"  SELECT * FROM {CATALOG}.gold.vw_avg_total_amount_per_month")
print(f"  SELECT * FROM {CATALOG}.gold.vw_avg_passengers_per_hour_may")

## Célula 6 — Bônus: Delta Time Travel

In [ ]:
spark.sql(f"DESCRIBE HISTORY {SILVER_TABLE}").select(
    "version","timestamp","operation","userName"
).show(10, truncate=False)
spark.sql(f'''
    SELECT 'versao_0' AS versao,
           ROUND(AVG(total_amount),2) AS media_total_amount,
           COUNT(*) AS total_registros
    FROM {SILVER_TABLE} VERSION AS OF 0
    UNION ALL
    SELECT 'atual',
           ROUND(AVG(total_amount),2),
           COUNT(*)
    FROM {SILVER_TABLE}
''').show()